[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/11_random_forest/90_exercises.ipynb)

# Bài tập — Chương 11: Random Forest (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/11_random_forest

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_classification_2d
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Viết công thức phương sai của trung bình $B$ model có phương sai $\sigma^2$, tương quan $\rho$. Bootstrap là gì, tỉ lệ OOB kỳ vọng? Random Forest thêm gì vào bagging?

✍️ Trả lời:

**Đáp án:** $\operatorname{Var} = \rho\sigma^2 + \frac{1 - \rho}{B}\sigma^2$. Bootstrap: rút có hoàn lại $n$ mẫu từ $n$ mẫu; mỗi điểm không được chọn với xác suất $(1 - 1/n)^n \to e^{-1} \approx 36.8\%$ (OOB). Random Forest = bagging cây + ở **mỗi split** chỉ xét một tập con ngẫu nhiên `max_features` feature → giảm tương quan giữa các cây.

## L1 — Giải thích bằng lời

**Câu 2.** Giải thích cho một quản lý: vì sao "trồng thêm cây" trong Random Forest không làm model overfit, trong khi "cho cây sâu hơn" thì có?

✍️ Trả lời:

**Gợi ý:** mỗi cây là một "ý kiến" độc lập-ish; thêm cây chỉ làm **trung bình** các ý kiến chính xác hơn (giảm dao động ngẫu nhiên), không làm model linh hoạt hơn — giống hỏi thêm người trong một cuộc khảo sát. Cho cây sâu hơn làm **từng ý kiến** bám sát dữ liệu train hơn (học thuộc nhiễu) — đó là tăng độ phức tạp. Thêm cây chỉ tốn thời gian/bộ nhớ; lợi ích bão hoà sau vài trăm cây.

## L2 — Tính tay

**Câu 3.** (a) 100 cây, mỗi cây variance 0.2, tương quan 0.25: variance của trung bình? (b) Variance tối thiểu khi số cây → ∞? (c) Với $n = 5$ mẫu, xác suất một mẫu là OOB của một cây?

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
var_avg = 0.25 * 0.2 + (1 - 0.25) / 100 * 0.2
var_limit = 0.25 * 0.2
p_oob_5 = (1 - 1 / 5) ** 5
### END CODE HERE ###

In [ ]:
check_close(var_avg, 0.0515, message="variance của trung bình")
check_close(var_limit, 0.05, message="giới hạn ρσ²")
check_close(p_oob_5, 0.32768, message="(4/5)^5")

**Câu 4.** Rừng 5 cây phân loại nhị phân, xác suất lớp 1 của từng cây cho một mẫu: `[0.9, 0.4, 0.45, 0.48, 0.3]`. (a) Dự đoán theo **soft voting** (trung bình xác suất, ngưỡng 0.5)? (b) Theo **hard voting** (mỗi cây bỏ phiếu ở ngưỡng 0.5)? Hai cách có cho cùng kết quả không? (scikit-learn dùng soft voting.)

In [ ]:
probs = np.array([0.9, 0.4, 0.45, 0.48, 0.3])
### START CODE HERE ### (≈ 2 dòng)
soft = int(probs.mean() >= 0.5)
hard = int((probs >= 0.5).sum() > len(probs) / 2)
### END CODE HERE ###

In [ ]:
check_equal(soft, 1, "soft voting: trung bình = 0.506 ≥ 0.5 → lớp 1")
check_equal(hard, 0, "hard voting: chỉ 1/5 cây bỏ phiếu lớp 1 → lớp 0")

## L3 — Code

**Bài 5.** Viết `bootstrap_oob(n, seed)` trả về `(idx, oob)`: chỉ số mẫu bootstrap và các chỉ số **không** xuất hiện trong mẫu (OOB). Kiểm tra tỉ lệ OOB ≈ 36.8%.

In [ ]:
def bootstrap_oob(n, seed=0):
    r = np.random.default_rng(seed)
    ### START CODE HERE ### (≈ 2 dòng)
    idx = r.integers(0, n, n)
    oob = np.setdiff1d(np.arange(n), idx)
    ### END CODE HERE ###
    return idx, oob

In [ ]:
fracs = [len(bootstrap_oob(5000, s)[1]) / 5000 for s in range(50)]
check_close(np.mean(fracs), np.exp(-1), tol=0.005, message="tỉ lệ OOB ≈ 1/e")
idx, oob = bootstrap_oob(100, 3)
check_true(len(np.intersect1d(idx, oob)) == 0, "OOB không giao với mẫu bootstrap")
check_equal(len(np.union1d(idx, oob)), 100, "hợp lại đủ mọi chỉ số")

**Bài 6.** Cài đặt `SimpleRandomForestRegressor` (bootstrap + `DecisionTreeRegressor(max_features=...)` + trung bình), có thuộc tính `oob_prediction_` (dự đoán OOB cho mỗi mẫu train, NaN nếu chưa từng là OOB) và `oob_r2_`. So sánh với `RandomForestRegressor(oob_score=True)`.

In [ ]:
class SimpleRandomForestRegressor:
    def __init__(self, n_estimators=100, max_features=1.0, min_samples_leaf=1, seed=0):
        self.n_estimators, self.max_features, self.min_samples_leaf, self.seed = n_estimators, max_features, min_samples_leaf, seed

    def fit(self, X, y):
        r = np.random.default_rng(self.seed)
        n = len(X)
        sums, counts = np.zeros(n), np.zeros(n)
        self.trees_ = []
        for _ in range(self.n_estimators):
            ### START CODE HERE ### (≈ 7 dòng)
            # STARTER: idx = None     # bootstrap n chỉ số
            # STARTER: oob = None     # chỉ số không có trong idx
            # STARTER: tree = None    # DecisionTreeRegressor(max_features=..., min_samples_leaf=..., random_state=int(r.integers(1 << 31))).fit(...)
            # STARTER: # thêm tree vào self.trees_; cộng dự đoán OOB vào sums[oob], tăng counts[oob]
            idx = r.integers(0, n, n)
            oob = np.setdiff1d(np.arange(n), idx)
            tree = DecisionTreeRegressor(max_features=self.max_features, min_samples_leaf=self.min_samples_leaf,
                                         random_state=int(r.integers(1 << 31))).fit(X[idx], y[idx])
            self.trees_.append(tree)
            sums[oob] += tree.predict(X[oob])
            counts[oob] += 1
            ### END CODE HERE ###
        self.oob_prediction_ = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
        ok = counts > 0
        self.oob_r2_ = 1 - np.sum((y[ok] - self.oob_prediction_[ok]) ** 2) / np.sum((y[ok] - y[ok].mean()) ** 2)
        return self

    def predict(self, X):
        ### START CODE HERE ### (≈ 1 dòng)
        # STARTER: return None    # trung bình dự đoán của mọi cây
        return np.mean([t.predict(X) for t in self.trees_], axis=0)
        ### END CODE HERE ###

In [ ]:
houses = pd.read_csv(repo_path("data", "sample", "house_prices.csv"))
Xh = pd.get_dummies(houses.drop(columns=["price_billion_vnd"]), columns=["district"], dtype=float).to_numpy()
yh = houses["price_billion_vnd"].to_numpy()
ours = SimpleRandomForestRegressor(n_estimators=200, max_features=0.5, min_samples_leaf=2).fit(Xh, yh)
sk = RandomForestRegressor(n_estimators=200, max_features=0.5, min_samples_leaf=2, oob_score=True, random_state=0).fit(Xh, yh)
print(f"OOB R² scratch = {ours.oob_r2_:.4f} | sklearn = {sk.oob_score_:.4f}")
check_close(ours.oob_r2_, sk.oob_score_, tol=0.01, message="OOB R² ≈ sklearn")

## L4 — Debug

**Bài 7.** Bạn của bạn cài "Random Forest" như dưới đây nhưng nó không tốt hơn **một cây** là bao. Tìm **2 lỗi** khiến các cây giống hệt nhau: (1) không bootstrap; (2) chọn feature ngẫu nhiên **một lần cho cả rừng** (và cùng `random_state`). Sửa trong `fixed_forest_proba`.

In [ ]:
Xd, yd = make_classification_2d("moons", n=600, noise=0.35, seed=12)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.4, random_state=0)


def buggy_forest_proba(X_tr, y_tr, X_te, n_trees=100):
    trees = [DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr) for _ in range(n_trees)]   # 🐛 cùng dữ liệu, cùng seed
    return np.mean([t.predict_proba(X_te)[:, 1] for t in trees], axis=0)


def fixed_forest_proba(X_tr, y_tr, X_te, n_trees=100, seed=0):
    r = np.random.default_rng(seed)
    probs = []
    for _ in range(n_trees):
        ### START CODE HERE ### (≈ 3 dòng)
        # STARTER: idx = None     # bootstrap
        # STARTER: tree = None    # cây với max_features ngẫu nhiên MỖI SPLIT và random_state khác nhau mỗi cây
        # STARTER: # probs.append(xác suất lớp 1 của tree trên X_te)
        idx = r.integers(0, len(X_tr), len(X_tr))
        tree = DecisionTreeClassifier(max_features=1, random_state=int(r.integers(1 << 31))).fit(X_tr[idx], y_tr[idx])
        probs.append(tree.predict_proba(X_te)[:, 1])
        ### END CODE HERE ###
    return np.mean(probs, axis=0)

In [ ]:
single_auc = roc_auc_score(yd_te, DecisionTreeClassifier(random_state=0).fit(Xd_tr, yd_tr).predict_proba(Xd_te)[:, 1])
buggy_auc = roc_auc_score(yd_te, buggy_forest_proba(Xd_tr, yd_tr, Xd_te))
fixed_auc = roc_auc_score(yd_te, fixed_forest_proba(Xd_tr, yd_tr, Xd_te))
print(f"1 cây = {single_auc:.4f} | rừng lỗi = {buggy_auc:.4f} | rừng đã sửa = {fixed_auc:.4f}")
check_close(buggy_auc, single_auc, tol=1e-9, message="rừng lỗi = đúng 1 cây (100 bản sao)")
check_greater(fixed_auc, single_auc + 0.05, "rừng đã sửa tốt hơn hẳn 1 cây")

## L5 — Phân tích

**Bài 8.** Trên dữ liệu `moons` (n=400, noise=0.35) với 8 feature **nhiễu** thêm vào (tổng 10 feature), vẽ ROC-AUC (5-fold CV) của RF theo `max_features` ∈ {1, 2, 3, 5, 10} và theo `min_samples_leaf` ∈ {1, 5, 20, 50} (heatmap). Lưu ma trận kết quả vào `auc_grid` (hàng = max_features, cột = min_samples_leaf). Nhận xét.

In [ ]:
Xm, ym = make_classification_2d("moons", n=400, noise=0.35, seed=6)
Xm = np.hstack([Xm, rng.normal(size=(len(Xm), 8))])
mf_values, leaf_values = [1, 2, 3, 5, 10], [1, 5, 20, 50]
auc_grid = np.zeros((len(mf_values), len(leaf_values)))
### START CODE HERE ### (≈ 4 dòng)
# STARTER: # với mỗi (i, mf) và (j, leaf): auc_grid[i, j] = ROC-AUC 5-fold CV của RandomForestClassifier(n_estimators=150, ..., random_state=0)
for i, mf in enumerate(mf_values):
    for j, leaf in enumerate(leaf_values):
        model = RandomForestClassifier(n_estimators=150, max_features=mf, min_samples_leaf=leaf, random_state=0)
        auc_grid[i, j] = cross_val_score(model, Xm, ym, cv=5, scoring="roc_auc").mean()
### END CODE HERE ###
plt.imshow(auc_grid, cmap="viridis")
plt.xticks(range(len(leaf_values)), leaf_values)
plt.yticks(range(len(mf_values)), mf_values)
plt.xlabel("min_samples_leaf")
plt.ylabel("max_features")
for i in range(len(mf_values)):
    for j in range(len(leaf_values)):
        plt.text(j, i, f"{auc_grid[i, j]:.3f}", ha="center", va="center", color="w")
plt.colorbar(label="ROC-AUC (CV)")
plt.show()

In [ ]:
check_greater(auc_grid[-1].max(), auc_grid[0].max(), "nhiều feature nhiễu → max_features lớn hơn tốt hơn max_features=1")

✍️ Nhận xét:

**Gợi ý:** với 8/10 feature là nhiễu, `max_features = 1` thường buộc split chọn feature nhiễu → cây rất yếu; tăng `max_features` để mỗi split có cơ hội thấy feature có ích. Khi tỉ lệ feature nhiễu cao, `max_features` tối ưu lớn hơn mặc định `sqrt`. `min_samples_leaf` vừa phải làm xác suất mượt hơn và tăng ROC-AUC; quá lớn → underfit (moons cần ranh giới cong).

## L6 — Thực tế

**Bài 9.** Model churn với Random Forest: one-hot cột phân loại của `customer_churn.csv`, chọn hyperparameter bằng **OOB score** (không dùng CV): thử `max_features` ∈ {2, 4, "sqrt", 8} × `min_samples_leaf` ∈ {1, 5, 20}, chọn cấu hình có **OOB ROC-AUC** cao nhất (train với `oob_score=True`, tính `roc_auc_score` trên `oob_decision_function_[:, 1]`). Train lại trên toàn bộ train, báo cáo ROC-AUC test (`test_auc`). Mục tiêu ≥ 0.80. In 5 feature có permutation importance cao nhất trên test.

In [ ]:
from sklearn.inspection import permutation_importance

df = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
Xc = pd.get_dummies(df.drop(columns=["customer_id", "churn"]), dtype=float)
yc = df["churn"].to_numpy()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=0, stratify=yc)
### START CODE HERE ### (≈ 9 dòng)
results = {}
for mf in [2, 4, "sqrt", 8]:
    for leaf in [1, 5, 20]:
        m = RandomForestClassifier(n_estimators=300, max_features=mf, min_samples_leaf=leaf, oob_score=True,
                                   random_state=0).fit(Xc_tr, yc_tr)
        results[(mf, leaf)] = roc_auc_score(yc_tr, m.oob_decision_function_[:, 1])
best_cfg = max(results, key=results.get)
best_rf = RandomForestClassifier(n_estimators=300, max_features=best_cfg[0], min_samples_leaf=best_cfg[1], random_state=0).fit(Xc_tr, yc_tr)
test_auc = roc_auc_score(yc_te, best_rf.predict_proba(Xc_te)[:, 1])
perm = permutation_importance(best_rf, Xc_te, yc_te, n_repeats=5, random_state=0, scoring="roc_auc")
### END CODE HERE ###
print("cấu hình tốt nhất (OOB):", best_cfg, f"| ROC-AUC test = {test_auc:.4f}")
print(pd.Series(perm.importances_mean, index=Xc.columns).nlargest(5).round(4))

In [ ]:
check_greater(test_auc, 0.80, "ROC-AUC test ≥ 0.80")

## L7 — Nâng cao

**Bài 10.** Kiểm chứng công thức $\operatorname{Var}(\bar f_B) = \rho\sigma^2 + \frac{1 - \rho}{B}\sigma^2$ **trên Random Forest thật**. Lưu ý: $\rho$ là tương quan giữa hai cây **train trên cùng một tập dữ liệu**, xét qua sự ngẫu nhiên của **cả** tập dữ liệu lẫn bootstrap/feature — nên cần nhiều tập train. Code dưới sinh $R = 40$ tập train từ `make_friedman1`, mỗi tập train một rừng $T = 40$ cây, lưu dự đoán vào `P` có shape $(R, T, \text{số điểm})$ cho hai cấu hình: Random Forest (`max_features="sqrt"`) và bagging (`max_features=1.0`).

Viết `rho_sigma(P)` trả về $(\sigma^2, \rho)$:
- $\sigma^2$ = phương sai dự đoán của **một cây** (trung bình bình phương độ lệch khỏi trung bình chung tại mỗi điểm, lấy trung bình qua mọi điểm);
- $\rho\sigma^2$ = hiệp phương sai trung bình giữa **hai cây khác nhau cùng một tập train**. Gợi ý: với $S = \sum_t d_t$ (tổng độ lệch của $T$ cây), $\sum_{i \ne j} d_i d_j = S^2 - \sum_t d_t^2$.

Rồi so sánh variance dự đoán bởi công thức với variance **đo được** của trung bình $B$ cây (qua $R$ tập train), $B \in \{1, 5, 10, 40\}$.

In [ ]:
from sklearn.datasets import make_friedman1

X_query, _ = make_friedman1(n_samples=200, n_features=10, noise=0, random_state=123)
R, T = 40, 40
P_all = {}
for mf in ["sqrt", 1.0]:
    P = np.zeros((R, T, len(X_query)))
    for r in range(R):
        Xr, yr = make_friedman1(n_samples=500, n_features=10, noise=1.0, random_state=r)
        f = RandomForestRegressor(n_estimators=T, max_features=mf, random_state=r).fit(Xr, yr)
        P[r] = [t.predict(X_query) for t in f.estimators_]
    P_all[mf] = P


def rho_sigma(P):
    R, T, _ = P.shape
    ### START CODE HERE ### (≈ 6 dòng)
    dev = P - P.mean(axis=(0, 1))                       # độ lệch khỏi trung bình chung tại mỗi điểm
    sigma2 = (dev**2).mean()
    S = dev.sum(axis=1)                                 # (R, điểm)
    SQ = (dev**2).sum(axis=1)
    cov = ((S**2 - SQ) / (T * (T - 1))).mean()          # trung bình d_i d_j, i ≠ j
    rho = cov / sigma2
    ### END CODE HERE ###
    return sigma2, rho


results10 = {}
for mf, P in P_all.items():
    sigma2, rho = rho_sigma(P)
    print(f"max_features = {mf}: σ² = {sigma2:.3f}, ρ = {rho:.3f}")
    for B in [1, 5, 10, 40]:
        predicted = rho * sigma2 + (1 - rho) / B * sigma2
        measured = P[:, :B].mean(axis=1).var(axis=0, ddof=1).mean()
        results10[(mf, B)] = (predicted, measured)
        print(f"   B = {B:2d}: công thức = {predicted:.3f} | đo được = {measured:.3f}")

In [ ]:
for (mf, B), (pred, meas) in results10.items():
    check_less(abs(pred - meas) / meas, 0.12, f"{mf}, B = {B}: công thức khớp đo đạc")
check_greater(rho_sigma(P_all[1.0])[1], rho_sigma(P_all["sqrt"])[1], "bagging (max_features=1.0) có ρ lớn hơn Random Forest")

✍️ Nhận xét: vì sao với $B = 40$ Random Forest có variance thấp hơn bagging dù $\sigma^2$ của từng cây lớn hơn?

**Gợi ý:** khi $B$ lớn, số hạng $\frac{1 - \rho}{B}\sigma^2$ nhỏ dần và variance tiến về **sàn** $\rho\sigma^2$. Random Forest có $\sigma^2$ lớn hơn (cây yếu hơn do chỉ xét $\sqrt d$ feature) nhưng $\rho$ nhỏ hơn nhiều → sàn $\rho\sigma^2$ thấp hơn bagging. Đó chính là lý do random feature subset hiệu quả.